In [ ]:
!pip install datasets scikit-learn rank_bm25 sentence-transformers nltk pytrec_eval gradio matplotlib pandas

In [ ]:
from huggingface_hub import snapshot_download
import pandas as pd, os

LANGS = ['ar','bn','en','es','fa','fi','fr','hi','id','ja','ko','ru','sw','te','th','zh']

path = snapshot_download(
    repo_id="miracl/miracl",
    repo_type="dataset",
    allow_patterns=["miracl-v1.0-*/topics/*", "miracl-v1.0-*/qrels/*"],
)
print(path)
print(os.listdir(f"{path}/miracl-v1.0-hi/topics"))
print(os.listdir(f"{path}/miracl-v1.0-hi/qrels"))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 97 files:   0%|          | 0/97 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/datasets--miracl--miracl/snapshots/5be20db9509754dadad47689368639fcec739c00
['topics.miracl-v1.0-hi-dev.tsv', 'topics.miracl-v1.0-hi-test-b.tsv', 'topics.miracl-v1.0-hi-train.tsv']
['qrels.miracl-v1.0-hi-train.tsv', 'qrels.miracl-v1.0-hi-dev.tsv']


In [ ]:
def load_split(lang, split="dev"):
    base = f"{path}/miracl-v1.0-{lang}"
    topics = pd.read_csv(
        f"{base}/topics/topics.miracl-v1.0-{lang}-{split}.tsv",
        sep="\t", header=None, names=["qid", "query"],
        dtype=str, quoting=3,          # quoting=3: don't treat " as special
    )
    qrels = pd.read_csv(
        f"{base}/qrels/qrels.miracl-v1.0-{lang}-{split}.tsv",
        sep="\t", header=None, names=["qid", "q0", "docid", "rel"],
        dtype={"qid": str, "docid": str, "rel": int},
    )
    return topics, qrels

data = {lang: load_split(lang) for lang in LANGS}

In [ ]:
rows = []
for lang, (topics, qrels) in data.items():
    rows.append({
        "lang": lang,
        "queries": len(topics),
        "judgments": len(qrels),
        "relevant": (qrels.rel == 1).sum(),
        "non_relevant": (qrels.rel == 0).sum(),
        "avg_relevant_per_query": round((qrels.rel == 1).sum() / len(topics), 2),
        "unique_docs": qrels.docid.nunique(),
    })
stats = pd.DataFrame(rows).set_index("lang")
print(stats)

      queries  judgments  relevant  non_relevant  avg_relevant_per_query  \
lang                                                                       
ar       2896      29197      5658         23539                    1.95   
bn        411       4206       863          3343                    2.10   
en        799       8350      2326          6024                    2.91   
es        648       6443      2987          3456                    4.61   
fa        632       6571      1314          5257                    2.08   
fi       1271      12008      2447          9561                    1.93   
fr        343       3429       731          2698                    2.13   
hi        350       3494       752          2742                    2.15   
id        960       9668      3088          6580                    3.22   
ja        860       8354      1790          6564                    2.08   
ko        213       3057       547          2510                    2.57   
ru       125

In [ ]:
topics, qrels = data["hi"]
print(topics.head(5).to_string())
print(qrels.head(10).to_string())

         qid                                  query
0  1033752#0           कांग्रेस दल का नेता कौन है ?
1   914952#0  पाकिस्तान में कोनसा धर्म सबसे बढ़ा है?
2   478739#0              गूगल की खोज किसने की थी ?
3   231611#0           एलोन मस्क का जन्म कब हुआ था?
4   191903#0          प्रथम विश्व युद्ध कब हुआ था ?
         qid  q0      docid  rel
0  1004960#0  Q0   69380#11    1
1  1004960#0  Q0   555530#0    1
2  1004960#0  Q0   555530#1    0
3  1004960#0  Q0   432721#1    0
4  1004960#0  Q0  496800#30    0
5  1004960#0  Q0  174163#10    0
6  1004960#0  Q0    34782#7    0
7  1004960#0  Q0  179523#76    0
8  1004960#0  Q0    53350#4    0
9  1004960#0  Q0   538245#7    0


In [ ]:
rows = []
for lang, (topics, qrels) in data.items():
    rel_per_q = qrels.groupby("qid").rel.sum()
    n_zero = (rel_per_q == 0).sum() + (len(topics) - qrels.qid.nunique())
    rows.append({"lang": lang, "queries_with_no_relevant": n_zero})
print(pd.DataFrame(rows).set_index("lang").T)

stats.to_csv("/content/stats_dev.csv")   # then copy it to your repo

lang                      ar  bn  en  es  fa  fi  fr  hi  id  ja  ko  ru  sw  \
queries_with_no_relevant   0   0   0   0   0   0   0   0   0   0   0   0   0   

lang                      te  th  zh  
queries_with_no_relevant   0   0   0  


In [ ]:
import re
from huggingface_hub import HfApi

files = HfApi().list_repo_files("miracl/miracl-corpus", repo_type="dataset")

def shard_key(f):
    return int(re.search(r"docs-(\d+)\.jsonl\.gz", f).group(1))

shards = {
    l: sorted([f for f in files
               if f.startswith(f"miracl-corpus-v1.0-{l}/docs-")], key=shard_key)
    for l in LANGS
}
for l in LANGS:
    print(l, len(shards[l]), "shards")

ar 5 shards
bn 1 shards
en 66 shards
es 21 shards
fa 5 shards
fi 4 shards
fr 30 shards
hi 2 shards
id 3 shards
ja 14 shards
ko 3 shards
ru 20 shards
sw 1 shards
te 2 shards
th 2 shards
zh 10 shards


In [ ]:
import gzip, json
from huggingface_hub import hf_hub_download

fp = hf_hub_download("miracl/miracl-corpus", shards["hi"][0],
                     repo_type="dataset", local_dir="/content/shards")
n = 0
with gzip.open(fp, "rt", encoding="utf-8") as fh:
    for i, line in enumerate(fh):
        d = json.loads(line)
        if i < 3:
            print(d)
        n += 1
print("passages in this shard:", n)

miracl-corpus-v1.0-hi/docs-0.jsonl.gz: reconstructing file:   0%|          |  0.00B / 95.7MB            

miracl-corpus-v1.0-hi/docs-0.jsonl.gz: downloading bytes:           |  0.00B            

{'docid': '10#0', 'title': 'हम होंगे कामयाब', 'text': 'हम होंगे कामयाब ( का गिरिजा कुमार माथुर द्वारा किया गया हिंदी भावानुवाद) एक प्रतिरोध गीत है। यह गीत बीसवीं सदी में नागरिक अधिकार आंदोलन का प्रधान स्वर बना। इस गीत को आमतौर पर "I\'ll Overcome Some Day" ("आई विल ओवरकम सम डे") से काव्यावतरित माना जाता है, जो चार्ल्स अल्बर्ट टिंडले द्वारा गाया गया था और जिसे 1900 में पहली बार प्रकाशित किया गया था।'}
{'docid': '10#1', 'title': 'हम होंगे कामयाब', 'text': 'Hum Honge Kamyab Lyrics'}
{'docid': '14#0', 'title': 'दैनिक पूजा', 'text': 'दैनिक पूजा विधि हिन्दू धर्म की कई उपासना पद्धतियों में से एक है। ये एक दैनिक कर्म है। विभिन्न देवताओं को प्रसन्न करने के लिये कई मन्त्र बताये गये हैं, जो लगभग सभी पुराणों से हैं। वैदिक मन्त्र यज्ञ और हवन के लिये होते हैं।'}
passages in this shard: 500000


In [ ]:
   from google.colab import drive
   drive.mount("/content/drive", force_remount=True)


Mounted at /content/drive


In [ ]:
import os, random
import pandas as pd

def build_corpus(lang, cap=50_000, seed=42):
    needed = set(data[lang][1].docid)            # every judged docid (dev qrels)
    k = max(cap - len(needed), 0)                # distractor budget
    rng = random.Random(seed)
    kept, reservoir, seen = {}, [], 0

    for shard in shards[lang]:
        fp = hf_hub_download("miracl/miracl-corpus", shard,
                             repo_type="dataset", local_dir="/content/shards")
        with gzip.open(fp, "rt", encoding="utf-8") as fh:
            for line in fh:
                d = json.loads(line)
                if d["docid"] in needed:
                    kept[d["docid"]] = d
                    continue
                seen += 1                        # reservoir sampling of distractors
                if len(reservoir) < k:
                    reservoir.append(d)
                else:
                    j = rng.randrange(seen)
                    if j < k:
                        reservoir[j] = d
        os.remove(fp)                            # free disk space

    print(f"{lang}: found {len(kept)}/{len(needed)} judged passages, "
          f"{len(reservoir)} distractors, scanned {seen + len(kept)} passages total")
    return pd.DataFrame(list(kept.values()) + reservoir)

from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/miracl/corpus", exist_ok=True)

import time; t = time.time()
corpus_hi = build_corpus("hi")
print("minutes:", round((time.time() - t) / 60, 1))
corpus_hi.to_parquet("/content/drive/MyDrive/miracl/corpus/hi.parquet")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


miracl-corpus-v1.0-hi/docs-1.jsonl.gz: reconstructing file:   0%|          |  0.00B / 1.16MB            

miracl-corpus-v1.0-hi/docs-1.jsonl.gz: downloading bytes:           |  0.00B            

hi: found 3342/3342 judged passages, 46658 distractors, scanned 506264 passages total
minutes: 0.2


In [ ]:
import threading

OUT = "/content/drive/MyDrive/miracl/corpus"
errors = {}

def run_languages(langs, cap=50_000):
    for lang in langs:
        out = f"{OUT}/{lang}.parquet"
        if os.path.exists(out):
            continue                      # resumable: finished languages are skipped
        try:
            df = build_corpus(lang, cap)
            assert df.docid.is_unique
            df.to_parquet(out)
        except Exception as e:
            errors[lang] = repr(e)

# small languages first, so you can start using them sooner
order = ["bn","sw","te","th","ko","id","fa","fi","ar","ja","zh","ru","es","fr","en"]
bg = threading.Thread(target=run_languages, args=(order,), daemon=True)
bg.start()

In [ ]:
done = sorted(f[:-8] for f in os.listdir(OUT) if f.endswith(".parquet"))
print(len(done), "of 16 done:", done, "| still running:", bg.is_alive())
print("errors:", errors)

14 of 16 done: ['ar', 'bn', 'es', 'fa', 'fi', 'hi', 'id', 'ja', 'ko', 'ru', 'sw', 'te', 'th', 'zh'] | still running: True
errors: {}


In [ ]:
import regex, unicodedata, functools, pickle
from collections import Counter
from nltk.stem.snowball import SnowballStemmer

TOKEN_RE = regex.compile(r"[\p{L}\p{M}\p{N}]+")
BIGRAM_LANGS = {"zh", "ja", "th"}
SNOWBALL = {"ar":"arabic","en":"english","es":"spanish",
            "fi":"finnish","fr":"french","ru":"russian"}

HI_SUFFIXES = sorted(["ाओं","ाएं","ाएँ","ियाँ","ियां","ियों","ेंगे","ोगे","ेगा","ेगी",
    "ाना","ाने","ाता","ाती","ाते","ता","ती","ते","ना","ने","ों","ें","ाँ","ां",
    "ी","ा","े","ो"], key=len, reverse=True)

def hindi_stem(w):
    for s in HI_SUFFIXES:
        if w.endswith(s) and len(w) - len(s) >= 2:
            return w[:-len(s)]
    return w

def clean(text):
    text = unicodedata.normalize("NFKC", text).lower()   # unify Unicode forms + lowercase
    return regex.sub(r"https?://\S+", " ", text)         # drop URLs

def tokenize(text, lang):
    toks = TOKEN_RE.findall(text)
    if lang in BIGRAM_LANGS:
        out = []
        for t in toks:
            out.extend([t] if len(t) == 1 else [t[i:i+2] for i in range(len(t) - 1)])
        return out
    return toks

def make_stemmer(lang):
    if lang in SNOWBALL:
        return functools.lru_cache(maxsize=None)(SnowballStemmer(SNOWBALL[lang]).stem)
    if lang == "hi":
        return functools.lru_cache(maxsize=None)(hindi_stem)
    return lambda w: w

def build_stopwords(token_lists, n=100):
    df = Counter()
    for toks in token_lists:
        df.update(set(toks))
    return {w for w, _ in df.most_common(n)}

def preprocess_corpus(df, lang):
    raw = [tokenize(clean(t), lang) for t in (df.title + " " + df.text)]
    stop = build_stopwords(raw)
    stem = make_stemmer(lang)
    docs = [[stem(w) for w in toks if w not in stop] for toks in raw]
    return docs, stop, stem

def preprocess_query(q, lang, stop, stem):
    return [stem(w) for w in tokenize(clean(q), lang) if w not in stop]

miracl-corpus-v1.0-fr/docs-0.jsonl.gz: reconstructing file:   0%|          |  0.00B / 81.0MB            

miracl-corpus-v1.0-fr/docs-0.jsonl.gz: downloading bytes:           |  0.00B            

In [ ]:
lang = "hi"
df = pd.read_parquet(f"{OUT}/{lang}.parquet")
docs, stop, stem = preprocess_corpus(df, lang)

print("sample stop-words:", list(stop)[:15])
print("BEFORE:", df.text.iloc[5][:150])
print("AFTER :", docs[5][:20])
print("avg tokens/doc:", sum(map(len, docs)) / len(docs))
print("query:", data[lang][0]["query"].iloc[0], "->",
      preprocess_query(data[lang][0]["query"].iloc[0], lang, stop, stem))

pickle.dump((df.docid.tolist(), docs, stop), open(f"{OUT}/{lang}_prep.pkl", "wb"))

miracl-corpus-v1.0-fr/docs-1.jsonl.gz: reconstructing file:   0%|          |  0.00B / 78.5MB            

miracl-corpus-v1.0-fr/docs-1.jsonl.gz: downloading bytes:           |  0.00B            

sample stop-words: ['लिया', 'अन्य', 'जब', 'है', 'इसके', 'करने', 'जाने', 'एक', 'भी', 'शामिल', 'जैसे', 'भारतीय', 'वाले', 'रूप', 'दो']
BEFORE: गणराज्य के पहले तीन चुनावों (१९५१–५२, १९५७, १९६२) में जवाहरलाल नेहरू के नेतृत्व में कॉंग्रेस ने आसान जीत पाई। १९६४ में नेहरू की मृत्यु के बाद लाल बहाद
AFTER : ['गणराज्य', 'तीन', 'चुनाव', '१९५१', '५२', '१९५७', '१९६२', 'जवाहरलाल', 'नेहरू', 'नेतृत्व', 'कॉंग्रेस', 'आसान', 'जीत', 'पाई', '१९६४', 'नेहरू', 'मृत्यु', 'लाल', 'बहादुर', 'शास्त्र']
avg tokens/doc: 42.07608
query: कांग्रेस दल का नेता कौन है ? -> ['कांग्रेस', 'दल', 'ने', 'कौन']


In [ ]:
def hindi_stem(w):
    for s in HI_SUFFIXES:
        if w.endswith(s) and len(w) - len(s) >= 3:      # was >= 2: keep stems at least 3 chars
            return w[:-len(s)]
    return w

def build_stopwords(token_lists, n=50):                 # was 100
    df = Counter()
    for toks in token_lists:
        df.update(set(toks))
    return {w for w, _ in df.most_common(n)}

def _norm(toks, stop, stem):
    out = []
    for w in toks:
        if w in stop:
            continue
        s = stem(w)
        if s not in stop:                               # drop stems that collapse into a stop-word
            out.append(s)
    return out

def preprocess_corpus(df, lang):
    raw = [tokenize(clean(t), lang) for t in (df.title + " " + df.text)]
    stop = build_stopwords(raw)
    stem = make_stemmer(lang)
    return [_norm(t, stop, stem) for t in raw], stop, stem

def preprocess_query(q, lang, stop, stem):
    return _norm(tokenize(clean(q), lang), stop, stem)

if os.path.exists(f"{OUT}/hi_prep.pkl"):
    os.remove(f"{OUT}/hi_prep.pkl")                     # old file used the buggy settings

In [ ]:
import numpy as np, scipy.sparse as sp, json
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

class TfidfRetriever:
    def __init__(self, texts):
        self.vec = TfidfVectorizer(tokenizer=str.split, lowercase=False,
                                   token_pattern=None, sublinear_tf=True, dtype=np.float32)
        self.D = self.vec.fit_transform(texts).T.tocsr()          # terms x docs
    def scores(self, qtexts):
        return (self.vec.transform(qtexts) @ self.D).toarray()

class BM25:
    def __init__(self, texts, k1=1.2, b=0.75):
        self.cv = CountVectorizer(tokenizer=str.split, lowercase=False, token_pattern=None)
        C = self.cv.fit_transform(texts).tocsr().astype(np.float32)
        N = C.shape[0]
        dl = np.asarray(C.sum(1)).ravel()
        df = np.bincount(C.indices, minlength=C.shape[1])
        idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
        rows = np.repeat(np.arange(N), np.diff(C.indptr))
        tf = C.data
        C.data = (tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[rows] / dl.mean())) * idf[C.indices]).astype(np.float32)
        self.W = C.T.tocsr()                                      # terms x docs
    def scores(self, qtexts):
        Q = self.cv.transform(qtexts).astype(np.float32)
        Q.data[:] = 1                                             # each query term counted once
        return (Q @ self.W).toarray()

def rank(retriever, qtexts, k=100, bs=250):
    out = []
    for i in range(0, len(qtexts), bs):
        s = retriever.scores(qtexts[i:i + bs])
        part = np.argpartition(-s, k, axis=1)[:, :k]
        order = np.argsort(-np.take_along_axis(s, part, 1), axis=1, kind="stable")
        out.append(np.take_along_axis(part, order, 1))
    return np.vstack(out)                                         # (n_queries, 100) doc indices

LEVELS = np.linspace(0, 1, 11)
DISC = 1 / np.log2(np.arange(2, 12))

def evaluate(top_idx, docids, qids, rel):
    m = {"P@10": [], "Recall@100": [], "MAP@100": [], "nDCG@10": []}
    pr = []
    for row, qid in zip(top_idx, qids):
        R = rel[qid]
        hits = np.array([docids[j] in R for j in row], dtype=float)
        cum = np.cumsum(hits)
        prec = cum / np.arange(1, len(hits) + 1)
        rec = cum / len(R)
        m["P@10"].append(hits[:10].mean())
        m["Recall@100"].append(rec[-1])
        m["MAP@100"].append((prec * hits).sum() / len(R))
        m["nDCG@10"].append((hits[:10] * DISC).sum() / DISC[:min(len(R), 10)].sum())
        pr.append([prec[rec >= r].max() if (rec >= r).any() else 0.0 for r in LEVELS])
    return {k: float(np.mean(v)) for k, v in m.items()}, np.mean(pr, axis=0)

RES = f"{OUT}/results"; os.makedirs(RES, exist_ok=True)

def prepare(lang):
    path = f"{OUT}/{lang}_prep.pkl"
    if not os.path.exists(path):
        df = pd.read_parquet(f"{OUT}/{lang}.parquet")
        docs, stop, _ = preprocess_corpus(df, lang)
        pickle.dump((df.docid.tolist(), docs, stop), open(path, "wb"))
    return pickle.load(open(path, "rb"))                          # docids, docs, stop

def run_sparse(lang):
    docids, docs, stop = prepare(lang)
    stem = make_stemmer(lang)
    texts = [" ".join(d) for d in docs]
    topics, qrels = data[lang]
    rel = qrels[qrels.rel == 1].groupby("qid").docid.apply(set).to_dict()
    qids = topics.qid.tolist()
    qtexts = [" ".join(preprocess_query(q, lang, stop, stem)) for q in topics["query"]]
    assert all(r <= set(docids) for r in rel.values()), "relevant docs missing from corpus!"
    out = {}
    for name, model in [("TF-IDF", TfidfRetriever(texts)), ("BM25", BM25(texts))]:
        top = rank(model, qtexts)
        metrics, pr = evaluate(top, docids, qids, rel)
        out[name] = {"metrics": metrics, "pr": pr.tolist()}
        np.save(f"{RES}/{lang}_{name}_top100.npy", top)           # needed later for RRF
        print(lang, name, {k: round(v, 3) for k, v in metrics.items()})
    json.dump(out, open(f"{RES}/{lang}_sparse.json", "w"))
    return out

# Test on Hindi first
print(preprocess_query(data["hi"][0]["query"].iloc[0], "hi", *prepare("hi")[2:], make_stemmer("hi")))
res_hi = run_sparse("hi")

miracl-corpus-v1.0-fr/docs-2.jsonl.gz: reconstructing file:   0%|          |  0.00B / 70.0MB            

miracl-corpus-v1.0-fr/docs-2.jsonl.gz: downloading bytes:           |  0.00B            

miracl-corpus-v1.0-fr/docs-3.jsonl.gz: reconstructing file:   0%|          |  0.00B / 66.9MB            

miracl-corpus-v1.0-fr/docs-3.jsonl.gz: downloading bytes:           |  0.00B            

['कांग्रेस', 'दल', 'नेत', 'कौन']
hi TF-IDF {'P@10': 0.123, 'Recall@100': 0.894, 'MAP@100': 0.324, 'nDCG@10': 0.39}
hi BM25 {'P@10': 0.149, 'Recall@100': 0.905, 'MAP@100': 0.421, 'nDCG@10': 0.5}


In [ ]:
import gc

def run_all_sparse():
    for lang in LANGS:
        p = f"{OUT}/{lang}.parquet"
        if os.path.exists(f"{RES}/{lang}_sparse.json"):
            continue                                     # already done
        if not os.path.exists(p) or time.time() - os.path.getmtime(p) < 60:
            print(lang, "corpus not ready yet"); continue
        t = time.time()
        try:
            run_sparse(lang)
            print(f"   ({lang} took {(time.time()-t)/60:.1f} min)")
        except Exception as e:
            print(lang, "FAILED:", repr(e))
        gc.collect()

run_all_sparse()

en corpus not ready yet


miracl-corpus-v1.0-fr/docs-4.jsonl.gz: reconstructing file:   0%|          |  0.00B / 59.5MB            

miracl-corpus-v1.0-fr/docs-4.jsonl.gz: downloading bytes:           |  0.00B            

miracl-corpus-v1.0-fr/docs-5.jsonl.gz: reconstructing file:   0%|          |  0.00B / 62.6MB            

miracl-corpus-v1.0-fr/docs-5.jsonl.gz: downloading bytes:           |  0.00B            

es TF-IDF {'P@10': 0.328, 'Recall@100': 0.955, 'MAP@100': 0.483, 'nDCG@10': 0.566}
es BM25 {'P@10': 0.366, 'Recall@100': 0.958, 'MAP@100': 0.557, 'nDCG@10': 0.651}
   (es took 0.4 min)
fr corpus not ready yet


In [ ]:
%%writefile /content/drive/MyDrive/miracl/common.py
import os, json, numpy as np, pandas as pd
from huggingface_hub import snapshot_download

LANGS = ['ar','bn','en','es','fa','fi','fr','hi','id','ja','ko','ru','sw','te','th','zh']
OUT = "/content/drive/MyDrive/miracl/corpus"
RES = f"{OUT}/results"
EMB = f"{OUT}/emb"
os.makedirs(RES, exist_ok=True); os.makedirs(EMB, exist_ok=True)

_path = snapshot_download(repo_id="miracl/miracl", repo_type="dataset",
        allow_patterns=["miracl-v1.0-*/topics/*", "miracl-v1.0-*/qrels/*"])

def load_split(lang, split="dev"):
    base = f"{_path}/miracl-v1.0-{lang}"
    topics = pd.read_csv(f"{base}/topics/topics.miracl-v1.0-{lang}-{split}.tsv", sep="\t",
                         header=None, names=["qid", "query"], dtype=str, quoting=3)
    qrels = pd.read_csv(f"{base}/qrels/qrels.miracl-v1.0-{lang}-{split}.tsv", sep="\t",
                        header=None, names=["qid", "q0", "docid", "rel"],
                        dtype={"qid": str, "docid": str, "rel": int})
    return topics, qrels

data = {l: load_split(l) for l in LANGS}

def get_rel(lang):
    q = data[lang][1]
    return q[q.rel == 1].groupby("qid").docid.apply(set).to_dict()

LEVELS = np.linspace(0, 1, 11)
DISC = 1 / np.log2(np.arange(2, 12))

def evaluate(top_idx, docids, qids, rel):
    m = {"P@10": [], "Recall@100": [], "MAP@100": [], "nDCG@10": []}
    pr = []
    for row, qid in zip(top_idx, qids):
        R = rel[qid]
        hits = np.array([docids[j] in R for j in row], dtype=float)
        cum = np.cumsum(hits)
        prec = cum / np.arange(1, len(hits) + 1)
        rec = cum / len(R)
        m["P@10"].append(hits[:10].mean())
        m["Recall@100"].append(rec[-1])
        m["MAP@100"].append((prec * hits).sum() / len(R))
        m["nDCG@10"].append((hits[:10] * DISC).sum() / DISC[:min(len(R), 10)].sum())
        pr.append([prec[rec >= r].max() if (rec >= r).any() else 0.0 for r in LEVELS])
    return {k: float(np.mean(v)) for k, v in m.items()}, np.mean(pr, axis=0)

Overwriting /content/drive/MyDrive/miracl/common.py


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip -q install sentence-transformers
import sys; sys.path.append("/content/drive/MyDrive/miracl")
from common import *
import torch, time, json
from sentence_transformers import SentenceTransformer

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


miracl-corpus-v1.0-fr/docs-6.jsonl.gz: reconstructing file:   0%|          |  0.00B / 69.2MB            

miracl-corpus-v1.0-fr/docs-6.jsonl.gz: downloading bytes:           |  0.00B            

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 97 files:   0%|          | 0/97 [00:00<?, ?it/s]

miracl-corpus-v1.0-fr/docs-7.jsonl.gz: reconstructing file:   0%|          |  0.00B / 68.2MB            

miracl-corpus-v1.0-fr/docs-7.jsonl.gz: downloading bytes:           |  0.00B            

In [ ]:
model = SentenceTransformer("intfloat/multilingual-e5-small", device="cuda")
model.half(); model.max_seq_length = 256          # fp16 + shorter inputs = faster

def embed_language(lang):
    out = f"{EMB}/{lang}_emb.npy"
    if os.path.exists(out): return
    df = pd.read_parquet(f"{OUT}/{lang}.parquet")
    texts = ("passage: " + df.title + " " + df.text).tolist()
    E = model.encode(texts, batch_size=128, normalize_embeddings=True,
                     convert_to_numpy=True, show_progress_bar=False)
    np.save(out, E.astype(np.float16))

def run_dense(lang):
    embed_language(lang)
    docids = pd.read_parquet(f"{OUT}/{lang}.parquet", columns=["docid"]).docid.tolist()
    E = torch.from_numpy(np.load(f"{EMB}/{lang}_emb.npy")).cuda()
    topics = data[lang][0]
    Q = model.encode(("query: " + topics["query"]).tolist(), batch_size=128,
                     normalize_embeddings=True, convert_to_tensor=True)
    top = (Q.to(E.dtype) @ E.T).topk(100, dim=1).indices.cpu().numpy()
    metrics, pr = evaluate(top, docids, topics.qid.tolist(), get_rel(lang))
    np.save(f"{RES}/{lang}_Dense_top100.npy", top)
    json.dump({"Dense": {"metrics": metrics, "pr": pr.tolist()}},
              open(f"{RES}/{lang}_dense.json", "w"))
    print(lang, {k: round(v, 3) for k, v in metrics.items()})

t = time.time(); run_dense("hi"); print("minutes:", round((time.time() - t) / 60, 1))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

hi {'P@10': 0.183, 'Recall@100': 0.982, 'MAP@100': 0.579, 'nDCG@10': 0.664}
minutes: 0.0


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
!ls /content/drive/MyDrive/miracl/corpus | head -50

Mounted at /content/drive
ar.parquet
ar_prep.pkl
bn.parquet
bn_prep.pkl
emb
es.parquet
es_prep.pkl
fa.parquet
fa_prep.pkl
fi.parquet
fi_prep.pkl
hi.parquet
hi_prep.pkl
id.parquet
id_prep.pkl
ja.parquet
ja_prep.pkl
ko.parquet
ko_prep.pkl
results
ru.parquet
ru_prep.pkl
sw.parquet
sw_prep.pkl
te.parquet
te_prep.pkl
th.parquet
th_prep.pkl
zh.parquet
zh_prep.pkl


In [ ]:
t = time.time()
for lang in LANGS:
    if not os.path.exists(f"{RES}/{lang}_dense.json"):
        run_dense(lang)
print("total minutes:", round((time.time() - t) / 60, 1))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/miracl/corpus/en.parquet'

In [ ]:
missing = [l for l in LANGS if not os.path.exists(f"{RES}/{l}_sparse.json")]
print("sparse results missing for:", missing)

In [ ]:
K_RRF = 60

def rrf(list_a, list_b, k=K_RRF, depth=100):
    fused = np.empty((len(list_a), depth), dtype=np.int64)
    for i, (a, b) in enumerate(zip(list_a, list_b)):
        sc = {}
        for r, d in enumerate(a, 1): sc[d] = sc.get(d, 0.0) + 1.0 / (k + r)
        for r, d in enumerate(b, 1): sc[d] = sc.get(d, 0.0) + 1.0 / (k + r)
        fused[i] = sorted(sc, key=sc.get, reverse=True)[:depth]
    return fused

def run_hybrid(lang):
    docids = pd.read_parquet(f"{OUT}/{lang}.parquet", columns=["docid"]).docid.tolist()
    top = rrf(np.load(f"{RES}/{lang}_BM25_top100.npy"),
              np.load(f"{RES}/{lang}_Dense_top100.npy"))
    topics = data[lang][0]
    metrics, pr = evaluate(top, docids, topics.qid.tolist(), get_rel(lang))
    np.save(f"{RES}/{lang}_Hybrid_top100.npy", top)
    json.dump({"Hybrid (RRF)": {"metrics": metrics, "pr": pr.tolist()}},
              open(f"{RES}/{lang}_hybrid.json", "w"))
    print(lang, {k: round(v, 3) for k, v in metrics.items()})

for lang in LANGS:
    run_hybrid(lang)

In [ ]:
MODELS = ["TF-IDF", "BM25", "Sentence Embedding", "Hybrid (RRF)"]
METRICS = ["P@10", "Recall@100", "MAP@100", "nDCG@10"]

rows, pr_curves = [], {}
for lang in LANGS:
    for part in ["sparse", "dense", "hybrid"]:
        d = json.load(open(f"{RES}/{lang}_{part}.json"))
        for name, v in d.items():
            name = {"Dense": "Sentence Embedding"}.get(name, name)
            rows.append({"lang": lang, "model": name, **v["metrics"]})
            pr_curves[(lang, name)] = v["pr"]
results = pd.DataFrame(rows)
results["model"] = pd.Categorical(results["model"], MODELS, ordered=True)
results = results.sort_values(["lang", "model"]).reset_index(drop=True)
results.to_csv(f"{RES}/comparison_table.csv", index=False)

ceil = {l: np.mean([min(len(R), 10) / 10 for R in get_rel(l).values()]) for l in LANGS}
print("P@10 ceiling per language:", {k: round(v, 2) for k, v in ceil.items()})

print("\nMACRO AVERAGE over 16 languages:")
macro = results.groupby("model", observed=True)[METRICS].mean().round(3)
print(macro)
macro.to_csv(f"{RES}/macro_average.csv")

for m in ["nDCG@10", "MAP@100"]:
    print(f"\n{m} by language:")
    print(results.pivot(index="lang", columns="model", values=m).round(3))

In [ ]:
import matplotlib.pyplot as plt
LV = np.linspace(0, 1, 11)
COL = {"TF-IDF": "tab:gray", "BM25": "tab:blue",
       "Sentence Embedding": "tab:orange", "Hybrid (RRF)": "tab:green"}

fig, axes = plt.subplots(4, 4, figsize=(18, 14), sharex=True, sharey=True)
for ax, lang in zip(axes.ravel(), LANGS):
    for m in MODELS:
        ax.plot(LV, pr_curves[(lang, m)], marker="o", ms=3, label=m, color=COL[m])
    ax.set_title(lang); ax.grid(alpha=.3)
axes[0, 0].legend(fontsize=8)
fig.supxlabel("Recall"); fig.supylabel("Interpolated precision")
plt.tight_layout(); plt.savefig(f"{RES}/pr_curves_all_languages.png", dpi=150); plt.show()

plt.figure(figsize=(7, 5))
for m in MODELS:
    plt.plot(LV, np.mean([pr_curves[(l, m)] for l in LANGS], axis=0),
             marker="o", label=m, color=COL[m])
plt.xlabel("Recall"); plt.ylabel("Interpolated precision")
plt.title("Precision-Recall, averaged over 16 languages")
plt.legend(); plt.grid(alpha=.3)
plt.savefig(f"{RES}/pr_curve_average.png", dpi=150); plt.show()

In [ ]:
docid_all, lang_all, offset, mats = [], [], {}, []
for l in LANGS:
    E = np.load(f"{EMB}/{l}_emb.npy")
    ids = pd.read_parquet(f"{OUT}/{l}.parquet", columns=["docid"]).docid.tolist()
    assert len(ids) == len(E)
    offset[l] = len(docid_all)
    docid_all += [f"{l}:{d}" for d in ids]
    lang_all += [l] * len(ids)
    mats.append(E)
U = torch.from_numpy(np.vstack(mats)).cuda(); del mats
lang_arr = np.array(lang_all)
print("unified index:", tuple(U.shape))

comp = pd.read_csv(f"{RES}/comparison_table.csv")
mono = comp[comp.model == "Sentence Embedding"].set_index("lang")

rows = []
for l in LANGS:
    topics = data[l][0]
    Q = model.encode(("query: " + topics["query"]).tolist(), batch_size=128,
                     normalize_embeddings=True, convert_to_tensor=True).to(U.dtype)
    top = np.vstack([(Q[i:i+256] @ U.T).topk(100, dim=1).indices.cpu().numpy()
                     for i in range(0, len(Q), 256)])
    rel = {q: {f"{l}:{d}" for d in s} for q, s in get_rel(l).items()}
    m, _ = evaluate(top, docid_all, topics.qid.tolist(), rel)
    rows.append({"lang": l,
                 "nDCG@10_unified": m["nDCG@10"], "nDCG@10_per_lang": mono.loc[l, "nDCG@10"],
                 "MAP@100_unified": m["MAP@100"], "MAP@100_per_lang": mono.loc[l, "MAP@100"],
                 "Recall@100_unified": m["Recall@100"],
                 "top10_same_language": (lang_arr[top[:, :10]] == l).mean()})

cross = pd.DataFrame(rows).set_index("lang")
cross["nDCG_drop"] = cross["nDCG@10_per_lang"] - cross["nDCG@10_unified"]
print(cross.round(3))
print("\nMEAN over 16 languages:\n", cross.mean().round(3))
cross.round(4).to_csv(f"{RES}/cross_lingual_unified.csv")

In [ ]:
dfs = {}
def xsearch(query, k=10, show=True):
    q = model.encode("query: " + query, normalize_embeddings=True,
                     convert_to_tensor=True).to(U.dtype)
    idx = (U @ q).topk(k).indices.cpu().tolist()
    langs = [lang_all[i] for i in idx]
    if show:
        print(f"\nQUERY: {query}")
        for r, i in enumerate(idx, 1):
            l = lang_all[i]
            if l not in dfs:
                dfs[l] = pd.read_parquet(f"{OUT}/{l}.parquet")
            row = dfs[l].iloc[i - offset[l]]
            print(f" {r:>2}. [{l}] {row.title} — {row.text[:110]!r}")
    return Counter(langs)

# The same question in three languages: where do the answers come from?
for q in ["Who is the leader of the Congress party in India?",
          "कांग्रेस दल का नेता कौन है ?",
          "インド国民会議の党首は誰ですか"]:
    print(dict(xsearch(q)))

# Another example
for q in ["When did the First World War start?", "प्रथम विश्व युद्ध कब हुआ था ?"]:
    print(dict(xsearch(q)))

In [ ]:
from collections import Counter

for q in ["Who is the leader of the Congress party in India?",
          "कांग्रेस दल का नेता कौन है ?",
          "インド国民会議の党首は誰ですか"]:
    print(dict(xsearch(q)))

for q in ["When did the First World War start?", "प्रथम विश्व युद्ध कब हुआ था ?"]:
    print(dict(xsearch(q)))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip -q install gradio
!mkdir -p /content/drive/MyDrive/miracl/app

In [ ]:
%%writefile /content/drive/MyDrive/miracl/app/prep.py
import regex, unicodedata, functools
from collections import Counter
from nltk.stem.snowball import SnowballStemmer

TOKEN_RE = regex.compile(r"[\p{L}\p{M}\p{N}]+")
BIGRAM_LANGS = {"zh", "ja", "th"}
SNOWBALL = {"ar": "arabic", "en": "english", "es": "spanish",
            "fi": "finnish", "fr": "french", "ru": "russian"}

HI_SUFFIXES = sorted(["ाओं","ाएं","ाएँ","ियाँ","ियां","ियों","ेंगे","ोगे","ेगा","ेगी",
    "ाना","ाने","ाता","ाती","ाते","ता","ती","ते","ना","ने","ों","ें","ाँ","ां",
    "ी","ा","े","ो"], key=len, reverse=True)

def hindi_stem(w):
    for s in HI_SUFFIXES:
        if w.endswith(s) and len(w) - len(s) >= 3:
            return w[:-len(s)]
    return w

def clean(text):
    text = unicodedata.normalize("NFKC", text).lower()
    return regex.sub(r"https?://\S+", " ", text)

def tokenize(text, lang):
    toks = TOKEN_RE.findall(text)
    if lang in BIGRAM_LANGS:
        out = []
        for t in toks:
            out.extend([t] if len(t) == 1 else [t[i:i+2] for i in range(len(t) - 1)])
        return out
    return toks

def make_stemmer(lang):
    if lang in SNOWBALL:
        return functools.lru_cache(maxsize=None)(SnowballStemmer(SNOWBALL[lang]).stem)
    if lang == "hi":
        return functools.lru_cache(maxsize=None)(hindi_stem)
    return lambda w: w

def build_stopwords(token_lists, n=50):
    df = Counter()
    for toks in token_lists:
        df.update(set(toks))
    return {w for w, _ in df.most_common(n)}

def _norm(toks, stop, stem):
    out = []
    for w in toks:
        if w in stop:
            continue
        s = stem(w)
        if s not in stop:
            out.append(s)
    return out

def preprocess_corpus(df, lang):
    raw = [tokenize(clean(t), lang) for t in (df.title + " " + df.text)]
    stop = build_stopwords(raw)
    stem = make_stemmer(lang)
    return [_norm(t, stop, stem) for t in raw], stop, stem

def preprocess_query(q, lang, stop, stem):
    return _norm(tokenize(clean(q), lang), stop, stem)

In [ ]:
%%writefile /content/drive/MyDrive/miracl/app/engines.py
import os, pickle
from collections import OrderedDict
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from huggingface_hub import snapshot_download
from prep import preprocess_query, make_stemmer

LANGS = ['ar','bn','en','es','fa','fi','fr','hi','id','ja','ko','ru','sw','te','th','zh']
BASE = os.environ.get("MIRACL_DIR", "/content/drive/MyDrive/miracl")
CORPUS = f"{BASE}/corpus"
RES = f"{CORPUS}/results"
EMB = f"{CORPUS}/emb"
MODEL_NAME = "intfloat/multilingual-e5-small"

class TfidfRetriever:
    def __init__(self, texts):
        self.vec = TfidfVectorizer(tokenizer=str.split, lowercase=False, token_pattern=None,
                                   sublinear_tf=True, dtype=np.float32)
        self.D = self.vec.fit_transform(texts).T.tocsr()
    def scores(self, qtexts):
        return (self.vec.transform(qtexts) @ self.D).toarray()

class BM25:
    def __init__(self, texts, k1=1.2, b=0.75):
        self.cv = CountVectorizer(tokenizer=str.split, lowercase=False, token_pattern=None)
        C = self.cv.fit_transform(texts).tocsr().astype(np.float32)
        N = C.shape[0]
        dl = np.asarray(C.sum(1)).ravel()
        df = np.bincount(C.indices, minlength=C.shape[1])
        idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
        rows = np.repeat(np.arange(N), np.diff(C.indptr))
        tf = C.data
        C.data = (tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[rows] / dl.mean())) * idf[C.indices]).astype(np.float32)
        self.W = C.T.tocsr()
    def scores(self, qtexts):
        Q = self.cv.transform(qtexts).astype(np.float32)
        Q.data[:] = 1
        return (Q @ self.W).toarray()

def top_k(scores, k):
    k = min(k, len(scores))
    part = np.argpartition(-scores, k - 1)[:k]
    return part[np.argsort(-scores[part], kind="stable")]

def rrf_fuse(rankings, k=60):
    sc = {}
    for r in rankings:
        for rank, d in enumerate(r, 1):
            sc[d] = sc.get(d, 0.0) + 1.0 / (k + rank)
    return sorted(sc.items(), key=lambda x: -x[1])

_model = None
def encode_query(q):
    global _model
    if _model is None:
        import torch
        from sentence_transformers import SentenceTransformer
        _model = SentenceTransformer(MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
        _model.max_seq_length = 256
    return _model.encode("query: " + q, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)

_dev_path = None
def load_dev(lang):
    global _dev_path
    if _dev_path is None:
        _dev_path = snapshot_download(repo_id="miracl/miracl", repo_type="dataset",
                                      allow_patterns=["miracl-v1.0-*/topics/*", "miracl-v1.0-*/qrels/*"])
    b = f"{_dev_path}/miracl-v1.0-{lang}"
    topics = pd.read_csv(f"{b}/topics/topics.miracl-v1.0-{lang}-dev.tsv", sep="\t", header=None,
                         names=["qid", "query"], dtype=str, quoting=3)
    qrels = pd.read_csv(f"{b}/qrels/qrels.miracl-v1.0-{lang}-dev.tsv", sep="\t", header=None,
                        names=["qid", "q0", "docid", "rel"], dtype={"qid": str, "docid": str, "rel": int})
    rel = qrels[qrels.rel == 1].groupby("qid").docid.apply(set).to_dict()
    return topics, rel

class LangEngine:
    def __init__(self, lang):
        self.lang = lang
        self.df = pd.read_parquet(f"{CORPUS}/{lang}.parquet")
        docids, docs, self.stop = pickle.load(open(f"{CORPUS}/{lang}_prep.pkl", "rb"))
        assert docids == self.df.docid.tolist(), "prep file does not match corpus"
        texts = [" ".join(d) for d in docs]
        self.tfidf, self.bm25 = TfidfRetriever(texts), BM25(texts)
        self.emb = np.load(f"{EMB}/{lang}_emb.npy").astype(np.float32)
        self.stem = make_stemmer(lang)
        self.topics, self.rel = load_dev(lang)
        self.q2qid = dict(zip(self.topics["query"].str.strip(), self.topics.qid))

    def _sparse(self, model, pq, k):
        s = model.scores([pq])[0]
        return [(int(i), float(s[i])) for i in top_k(s, k) if s[i] > 0]

    def _dense(self, q, k):
        s = self.emb @ encode_query(q)
        return [(int(i), float(s[i])) for i in top_k(s, k)]

    def search(self, query, model, k=10):
        pq = " ".join(preprocess_query(query, self.lang, self.stop, self.stem))
        if model == "TF-IDF":
            return self._sparse(self.tfidf, pq, k)
        if model == "BM25":
            return self._sparse(self.bm25, pq, k)
        if model == "Sentence Embedding":
            return self._dense(query, k)
        b = [i for i, _ in self._sparse(self.bm25, pq, 100)]      # Hybrid (RRF)
        d = [i for i, _ in self._dense(query, 100)]
        return rrf_fuse([b, d])[:k]

    def passage(self, i):
        r = self.df.iloc[i]
        return r.docid, r.title, r.text

_cache = OrderedDict()
MAX_LOADED = 4                                   # keep memory bounded
def get_engine(lang):
    if lang in _cache:
        _cache.move_to_end(lang)
        return _cache[lang]
    _cache[lang] = LangEngine(lang)
    while len(_cache) > MAX_LOADED:
        _cache.popitem(last=False)
    return _cache[lang]

# ---------- unified cross-lingual index (all 16 languages, embeddings only) ----------
_U = None; _U_meta = None; _dfs = {}
def get_unified():
    global _U, _U_meta
    if _U is None:
        import torch
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        dt = torch.float16 if dev == "cuda" else torch.float32
        mats, langs, rows = [], [], []
        for l in LANGS:
            E = np.load(f"{EMB}/{l}_emb.npy"); mats.append(E)
            langs += [l] * len(E); rows += range(len(E))
        _U = torch.from_numpy(np.vstack(mats)).to(device=dev, dtype=dt)
        _U_meta = (np.array(langs), np.array(rows))
    return _U, _U_meta

def search_all(query, k=10):
    import torch
    U, (L, R) = get_unified()
    q = torch.from_numpy(encode_query(query)).to(device=U.device, dtype=U.dtype)
    vals, idx = (U @ q).topk(k)
    out = []
    for v, i in zip(vals.tolist(), idx.tolist()):
        l = str(L[i])
        if l not in _dfs:
            _dfs[l] = pd.read_parquet(f"{CORPUS}/{l}.parquet")
        r = _dfs[l].iloc[int(R[i])]
        out.append((l, r.docid, r.title, r.text, v))
    return out

In [ ]:
%%writefile /content/drive/MyDrive/miracl/app/app.py
import html, json
import numpy as np, pandas as pd, gradio as gr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from engines import LANGS, RES, get_engine, search_all

NAMES = {"ar": "Arabic", "bn": "Bengali", "en": "English", "es": "Spanish", "fa": "Persian",
         "fi": "Finnish", "fr": "French", "hi": "Hindi", "id": "Indonesian", "ja": "Japanese",
         "ko": "Korean", "ru": "Russian", "sw": "Swahili", "te": "Telugu", "th": "Thai", "zh": "Chinese"}
LANG_CHOICES = [f"{l} - {NAMES[l]}" for l in LANGS]
MODELS = ["TF-IDF", "BM25", "Sentence Embedding", "Hybrid (RRF)"]
ALL = "Compare all four"
OWN = "(type your own)"

COMP = pd.read_csv(f"{RES}/comparison_table.csv").round(3)
MACRO = pd.read_csv(f"{RES}/macro_average.csv").round(3)

def code(choice):
    return choice.split(" - ")[0].strip()

def card(rank, docid, title, text, score, rel):
    ok = rel is not None and docid in rel
    border = "#2e9e5b" if ok else "rgba(128,128,128,.35)"
    bg = "rgba(46,158,91,.13)" if ok else "rgba(128,128,128,.07)"
    badge = (' <span style="background:#2e9e5b;color:#fff;border-radius:6px;padding:0 6px;'
             'font-size:.75em">✔ relevant (qrels)</span>') if ok else ""
    snip = html.escape(text[:320]) + ("…" if len(text) > 320 else "")
    return (f'<div style="border:1px solid {border};background:{bg};border-radius:10px;'
            f'padding:10px 12px;margin-bottom:8px">'
            f'<div style="display:flex;justify-content:space-between;font-size:.8em;opacity:.75">'
            f'<span>#{rank} · {html.escape(docid)}</span><span>score {score:.3f}</span></div>'
            f'<div style="font-weight:600;margin:3px 0">{html.escape(title)}{badge}</div>'
            f'<div style="font-size:.9em;line-height:1.45">{snip}</div></div>')

def run_search(lang_choice, query, model, k):
    query = (query or "").strip()
    if not query:
        return "<p>Please type a query or pick a sample query.</p>"
    eng = get_engine(code(lang_choice))
    qid = eng.q2qid.get(query)
    rel = eng.rel.get(qid) if qid else None
    note = ""
    if rel is not None:
        note = (f'<p style="font-size:.9em">This is a MIRACL dev query with {len(rel)} known relevant '
                f'passage(s). Green = judged relevant; unmarked = not judged relevant.</p>')
    cols = []
    for m in (MODELS if model == ALL else [model]):
        hits = eng.search(query, m, int(k))
        cards, n_rel = "", 0
        for r, (i, s) in enumerate(hits, 1):
            docid, title, text = eng.passage(i)
            n_rel += rel is not None and docid in rel
            cards += card(r, docid, title, text, s, rel)
        summary = (f' <span style="font-weight:400;font-size:.85em">({n_rel}/{len(hits)} relevant)</span>'
                   if rel is not None else "")
        cards = cards or "<i>No results (no searchable terms after preprocessing).</i>"
        cols.append(f'<div><div style="font-weight:700;margin-bottom:6px">{m}{summary}</div>{cards}</div>')
    grid = ('<div style="display:grid;gap:14px;grid-template-columns:repeat(auto-fit,minmax(300px,1fr))">'
            + "".join(cols) + "</div>")
    return note + grid

def run_cross(query, k):
    query = (query or "").strip()
    if not query:
        return "<p>Type a query in any language.</p>"
    hits = search_all(query, int(k))
    return "".join(card(r, f"[{NAMES[l]}] {d}", t, x, sc, None)
                   for r, (l, d, t, x, sc) in enumerate(hits, 1))

def on_language(choice):
    eng = get_engine(code(choice))
    samples = [OWN] + eng.topics["query"].head(40).tolist()
    return gr.Dropdown(choices=samples, value=OWN), ""

def on_sample(s):
    return "" if s == OWN else s

def table_for(l):
    return COMP if l == "All languages" else COMP[COMP["lang"] == l]

def pr_plot(choice):
    langs = LANGS if choice.startswith("Average") else [code(choice)]
    curves = {}
    for l in langs:
        for part in ("sparse", "dense", "hybrid"):
            for name, v in json.load(open(f"{RES}/{l}_{part}.json")).items():
                curves.setdefault("Sentence Embedding" if name == "Dense" else name, []).append(v["pr"])
    lv = np.linspace(0, 1, 11)
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for m in MODELS:
        ax.plot(lv, np.mean(curves[m], axis=0), marker="o", label=m)
    ax.set_xlabel("Recall"); ax.set_ylabel("Interpolated precision")
    ax.set_title("Precision-Recall: " + ("average of 16 languages" if len(langs) > 1 else NAMES[langs[0]]))
    ax.grid(alpha=.3); ax.legend()
    return fig



with gr.Blocks(title="MIRACL Multilingual Search") as demo:
    gr.Markdown("# Multilingual Search on MIRACL\nSearch Wikipedia passages in 16 languages with four ranking models.")
    with gr.Tab("Search"):
        with gr.Row():
            lang = gr.Dropdown(LANG_CHOICES, value=[c for c in LANG_CHOICES if c.startswith("hi")][0], label="Language")
            sample = gr.Dropdown([OWN], value=OWN, label="Sample dev queries")
        query = gr.Textbox(label="Query", placeholder="Type a question in the selected language")
        with gr.Row():
            model = gr.Radio(MODELS + [ALL], value=ALL, label="Ranking model")
            topk = gr.Slider(5, 20, value=10, step=1, label="Results per model")
        btn = gr.Button("Search", variant="primary")
        out = gr.HTML()
        lang.change(on_language, lang, [sample, query])
        sample.change(on_sample, sample, query)
        btn.click(run_search, [lang, query, model, topk], out)
        query.submit(run_search, [lang, query, model, topk], out)
        demo.load(on_language, lang, [sample, query])
    with gr.Tab("Cross-lingual"):
        gr.Markdown("One embedding index over **all 16 languages** (~800,000 passages). "
                    "Write the query in any language; results can come from any language. "
                    "The first search loads the index and takes a little longer.")
        cq = gr.Textbox(label="Query (any language)")
        ck = gr.Slider(5, 20, value=10, step=1, label="Results")
        cb = gr.Button("Search all languages", variant="primary")
        co = gr.HTML()
        cb.click(run_cross, [cq, ck], co)
        cq.submit(run_cross, [cq, ck], co)
    with gr.Tab("Results"):
        gr.Markdown("**Macro average over 16 languages**")
        gr.Dataframe(value=MACRO, interactive=False)
        gr.Markdown("**Per language** (note: P@10 is capped by the number of relevant passages per query, "
                    "which is only about 1 to 4.6 depending on the language)")
        flt = gr.Dropdown(["All languages"] + LANGS, value="All languages", label="Filter by language")
        tbl = gr.Dataframe(value=COMP, interactive=False)
        flt.change(table_for, flt, tbl)
    with gr.Tab("Precision-Recall"):
        prd = gr.Dropdown(["Average of all 16"] + LANG_CHOICES, value="Average of all 16", label="Language")
        plot = gr.Plot()
        prd.change(pr_plot, prd, plot)
        demo.load(pr_plot, prd, plot)


if __name__ == "__main__":
    for l in ("hi", "en"):
        get_engine(l)                      # warm up: builds indexes before the first user query
    demo.queue().launch(share=True)

In [ ]:
%%writefile /content/drive/MyDrive/miracl/app/app.py
import html, json
import numpy as np, pandas as pd, gradio as gr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from engines import LANGS, RES, get_engine, search_all

NAMES = {"ar": "Arabic", "bn": "Bengali", "en": "English", "es": "Spanish", "fa": "Persian",
         "fi": "Finnish", "fr": "French", "hi": "Hindi", "id": "Indonesian", "ja": "Japanese",
         "ko": "Korean", "ru": "Russian", "sw": "Swahili", "te": "Telugu", "th": "Thai", "zh": "Chinese"}
LANG_CHOICES = [f"{l} - {NAMES[l]}" for l in LANGS]
MODELS = ["TF-IDF", "BM25", "Sentence Embedding", "Hybrid (RRF)"]
ALL = "Compare all four"
OWN = "(type your own)"

COMP = pd.read_csv(f"{RES}/comparison_table.csv").round(3)
MACRO = pd.read_csv(f"{RES}/macro_average.csv").round(3)

def code(choice):
    return choice.split(" - ")[0].strip()

def card(rank, docid, title, text, score, rel):
    ok = rel is not None and docid in rel
    border = "#2e9e5b" if ok else "rgba(128,128,128,.35)"
    bg = "rgba(46,158,91,.13)" if ok else "rgba(128,128,128,.07)"
    badge = (' <span style="background:#2e9e5b;color:#fff;border-radius:6px;padding:0 6px;'
             'font-size:.75em">✔ relevant (qrels)</span>') if ok else ""
    snip = html.escape(text[:320]) + ("…" if len(text) > 320 else "")
    return (f'<div style="border:1px solid {border};background:{bg};border-radius:10px;'
            f'padding:10px 12px;margin-bottom:8px">'
            f'<div style="display:flex;justify-content:space-between;font-size:.8em;opacity:.75">'
            f'<span>#{rank} · {html.escape(docid)}</span><span>score {score:.3f}</span></div>'
            f'<div style="font-weight:600;margin:3px 0">{html.escape(title)}{badge}</div>'
            f'<div style="font-size:.9em;line-height:1.45">{snip}</div></div>')

def run_search(lang_choice, query, model, k):
    query = (query or "").strip()
    if not query:
        return "<p>Please type a query or pick a sample query.</p>"
    eng = get_engine(code(lang_choice))
    qid = eng.q2qid.get(query)
    rel = eng.rel.get(qid) if qid else None
    note = ""
    if rel is not None:
        note = (f'<p style="font-size:.9em">This is a MIRACL dev query with {len(rel)} known relevant '
                f'passage(s). Green = judged relevant; unmarked = not judged relevant.</p>')
    cols = []
    for m in (MODELS if model == ALL else [model]):
        hits = eng.search(query, m, int(k))
        cards, n_rel = "", 0
        for r, (i, s) in enumerate(hits, 1):
            docid, title, text = eng.passage(i)
            n_rel += rel is not None and docid in rel
            cards += card(r, docid, title, text, s, rel)
        summary = (f' <span style="font-weight:400;font-size:.85em">({n_rel}/{len(hits)} relevant)</span>'
                   if rel is not None else "")
        cards = cards or "<i>No results (no searchable terms after preprocessing).</i>"
        cols.append(f'<div><div style="font-weight:700;margin-bottom:6px">{m}{summary}</div>{cards}</div>')
    grid = ('<div style="display:grid;gap:14px;grid-template-columns:repeat(auto-fit,minmax(300px,1fr))">'
            + "".join(cols) + "</div>")
    return note + grid

def run_cross(query, k):
    query = (query or "").strip()
    if not query:
        return "<p>Type a query in any language.</p>"
    hits = search_all(query, int(k))
    return "".join(card(r, f"[{NAMES[l]}] {d}", t, x, sc, None)
                   for r, (l, d, t, x, sc) in enumerate(hits, 1))

def on_language(choice):
    eng = get_engine(code(choice))
    samples = [OWN] + eng.topics["query"].head(40).tolist()
    return gr.Dropdown(choices=samples, value=OWN), ""

def on_sample(s):
    return "" if s == OWN else s

def table_for(l):
    return COMP if l == "All languages" else COMP[COMP["lang"] == l]

def pr_plot(choice):
    langs = LANGS if choice.startswith("Average") else [code(choice)]
    curves = {}
    for l in langs:
        for part in ("sparse", "dense", "hybrid"):
            for name, v in json.load(open(f"{RES}/{l}_{part}.json")).items():
                curves.setdefault("Sentence Embedding" if name == "Dense" else name, []).append(v["pr"])
    lv = np.linspace(0, 1, 11)
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for m in MODELS:
        ax.plot(lv, np.mean(curves[m], axis=0), marker="o", label=m)
    ax.set_xlabel("Recall"); ax.set_ylabel("Interpolated precision")
    ax.set_title("Precision-Recall: " + ("average of 16 languages" if len(langs) > 1 else NAMES[langs[0]]))
    ax.grid(alpha=.3); ax.legend()
    return fig

with gr.Blocks(title="MIRACL Multilingual Search") as demo:
    gr.Markdown("# Multilingual Search on MIRACL\nSearch Wikipedia passages in 16 languages with four ranking models.")
    with gr.Tab("Search"):
        with gr.Row():
            lang = gr.Dropdown(LANG_CHOICES, value=[c for c in LANG_CHOICES if c.startswith("hi")][0], label="Language")
            sample = gr.Dropdown([OWN], value=OWN, label="Sample dev queries")
        query = gr.Textbox(label="Query", placeholder="Type a question in the selected language")
        with gr.Row():
            model = gr.Radio(MODELS + [ALL], value=ALL, label="Ranking model")
            topk = gr.Slider(5, 20, value=10, step=1, label="Results per model")
        btn = gr.Button("Search", variant="primary")
        out = gr.HTML()
        lang.change(on_language, lang, [sample, query])
        sample.change(on_sample, sample, query)
        btn.click(run_search, [lang, query, model, topk], out)
        query.submit(run_search, [lang, query, model, topk], out)
        demo.load(on_language, lang, [sample, query])
    with gr.Tab("Cross-lingual"):
        gr.Markdown("One embedding index over **all 16 languages** (~800,000 passages). "
                    "Write the query in any language; results can come from any language. "
                    "The first search loads the index and takes a little longer.")
        cq = gr.Textbox(label="Query (any language)")
        ck = gr.Slider(5, 20, value=10, step=1, label="Results")
        cb = gr.Button("Search all languages", variant="primary")
        co = gr.HTML()
        cb.click(run_cross, [cq, ck], co)
        cq.submit(run_cross, [cq, ck], co)
    with gr.Tab("Results"):
        gr.Markdown("**Macro average over 16 languages**")
        gr.Dataframe(value=MACRO, interactive=False)
        gr.Markdown("**Per language** (note: P@10 is capped by the number of relevant passages per query, "
                    "which is only about 1 to 4.6 depending on the language)")
        flt = gr.Dropdown(["All languages"] + LANGS, value="All languages", label="Filter by language")
        tbl = gr.Dataframe(value=COMP, interactive=False)
        flt.change(table_for, flt, tbl)
    with gr.Tab("Precision-Recall"):
        prd = gr.Dropdown(["Average of all 16"] + LANG_CHOICES, value="Average of all 16", label="Language")
        plot = gr.Plot()
        prd.change(pr_plot, prd, plot)
        demo.load(pr_plot, prd, plot)

if __name__ == "__main__":
    for l in ("hi", "en"):
        get_engine(l)                      # warm up: builds indexes before the first user query
    demo.queue().launch(share=True)

In [ ]:
!cd /content/drive/MyDrive/miracl/app && python -c "from engines import *; e = get_engine('hi'); print(e.search('कांग्रेस दल का नेता कौन है', 'Hybrid (RRF)', 3))"

In [ ]:
import subprocess, time, re
from IPython.display import display, HTML

# 1. stop any old copies of the app
subprocess.run("pkill -f 'app/app[.]py'", shell=True)
time.sleep(2)

# 2. start the app in the background, unbuffered so the link shows up in the log
log_path = "/content/app.log"
log = open(log_path, "w")
proc = subprocess.Popen(
    ["python", "-u", "/content/drive/MyDrive/miracl/app/app.py"],
    stdout=log, stderr=subprocess.STDOUT,
    cwd="/content/drive/MyDrive/miracl/app",
)

# 3. wait for the public link (the hi/en indexes build first, so allow a few minutes)
link = None
for _ in range(120):                         # up to ~10 minutes
    time.sleep(5)
    txt = open(log_path).read()
    m = re.search(r"https://[\w-]+\.gradio\.live", txt)
    if m:
        link = m.group(0)
        break
    if proc.poll() is not None:              # app crashed: show why
        print(txt[-2000:])
        break

if link:
    display(HTML(f'<h3>App is live: <a href="{link}" target="_blank">{link}</a></h3>'))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip -q install gradio sentence-transformers nltk regex pyarrow

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


miracl-corpus-v1.0-fr/docs-12.jsonl.gz: reconstructing file:   0%|          |  0.00B / 66.8MB            

miracl-corpus-v1.0-fr/docs-12.jsonl.gz: downloading bytes:           |  0.00B            

In [ ]:
import subprocess
subprocess.run("pkill -f 'app/app[.]py'", shell=True)
subprocess.Popen(
    "nohup python -u /content/drive/MyDrive/miracl/app/app.py > /content/app.log 2>&1 &",
    shell=True, cwd="/content/drive/MyDrive/miracl/app")
print("started, check the log in 2 to 5 minutes")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/miracl/app'

In [ ]:
import re
txt = open("/content/app.log").read()
m = re.search(r"https://[\w-]+\.gradio\.live", txt)
print(m.group(0) if m else txt[-1500:])

FileNotFoundError: [Errno 2] No such file or directory: '/content/app.log'

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os
OUT = "/content/drive/MyDrive/miracl/corpus"
LANGS = ['ar','bn','en','es','fa','fi','fr','hi','id','ja','ko','ru','sw','te','th','zh']
print("missing:", [l for l in LANGS if not (os.path.exists(f"{OUT}/{l}.parquet")
      and os.path.exists(f"{OUT}/{l}_prep.pkl") and os.path.exists(f"{OUT}/emb/{l}_emb.npy"))])

Mounted at /content/drive
missing: []


In [ ]:
!pip -q install gradio sentence-transformers regex

In [ ]:
!python /content/drive/MyDrive/miracl/app/app.py

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 97 files: 100% 97/97 [00:00<00:00, 1214.26it/s]
Download complete:           |  0.00B            
Download complete:           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://6226cfb204d1ef910b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Loading weights: 100% 199/199 [00:00<00:00, 5400.48it/s]
